# Faymoos Premium AI Lab

This notebook builds a **measurable hybrid AI layer** for Faymoos rather than inventing a single "AI accuracy" number.

Production design:

1. multilingual visitor-intent classification with out-of-scope rejection,
2. semantic retrieval of real creator projects/testimonials,
3. conversion ranking learned only from real Faymoos events,
4. grounded LLM generation that is not allowed to invent proof,
5. separate evaluation gates for every subsystem.

The repository includes a multilingual bootstrap dataset. Public datasets are used as representation/robustness candidates; first-party data remains the real source of product accuracy.


## Dataset plan

Known public candidates used by this lab:

- **Amazon MASSIVE** (`AmazonScience/massive`) for multilingual intent representation.
- **CLINC150/OOS** as an optional out-of-scope benchmark.
- **GoEmotions** only if message tone triage is later required.
- **ADE20K / SUN RGB-D** are vision candidates for Smart Space object/hotspot research, not for panorama geometry.

Before commercial deployment, verify the current dataset card, availability and license. This notebook never mixes public intent labels with fake Faymoos conversion labels.


In [ ]:
!pip -q install datasets sentence-transformers scikit-learn pandas numpy joblib lightgbm matplotlib


In [ ]:
from pathlib import Path
import json, os, random, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    average_precision_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED); np.random.seed(SEED)
ARTIFACTS = Path('ai/artifacts'); ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA = Path('ai/data')
print('Artifacts:', ARTIFACTS.resolve())


## 1. Public multilingual benchmark smoke test

MASSIVE contains many intents unrelated to Faymoos. We do **not** treat its intent score as Faymoos product accuracy. It is used to test whether an encoder carries useful multilingual intent structure.

For a Colab run, evaluate more rows/locales if GPU/RAM permits.


In [ ]:
MASSIVE_CONFIGS = ['en-US', 'fr-FR', 'ar-SA']
frames = []
for config in MASSIVE_CONFIGS:
    try:
        ds = load_dataset('AmazonScience/massive', config, split='train[:4000]')
        frames.append(pd.DataFrame({'text': ds['utt'], 'label': ds['intent'], 'locale': config}))
        print(config, len(ds))
    except Exception as exc:
        warnings.warn(f'Skipping {config}: {exc}')

massive_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['text','label','locale'])
massive_df.head()


## 2. Faymoos-specific multilingual intent dataset

The included bootstrap labels are:

`hire`, `book_consultation`, `request_quote`, `buy`, `view_portfolio`, `collaborate`, `learn_more`, `other`.

The bootstrap set is deliberately small and readable so labels can be audited. It must later be augmented with privacy-safe real visitor queries. Never report the bootstrap score as a production KPI.


In [ ]:
f = pd.read_csv(DATA / 'faymoos_intents.csv')
print(f.shape)
display(pd.crosstab(f.intent, f.locale))
f.sample(min(12, len(f)), random_state=SEED)


### Offline lexical baseline
A word+character TF-IDF baseline is useful because it gives us something cheap to beat. It is expected to be weak across languages, especially when one language is absent from training. Never confuse a bootstrap baseline with the production target.


In [ ]:
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score

lexical = Pipeline([
    ('features', FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, strip_accents='unicode')),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), sublinear_tf=True)),
    ])),
    ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', C=4.0)),
])
cv = StratifiedKFold(n_splits=6, shuffle=True, random_state=SEED)
lex_scores = cross_val_score(lexical, f.text, f.intent, cv=cv, scoring='f1_macro')
print('TF-IDF macro-F1:', round(float(lex_scores.mean()), 4), '+/-', round(float(lex_scores.std()), 4))


## 3. Benchmark several multilingual encoders

We benchmark multiple encoders using the same classifier and split. The selected model is the one with the best **held-out macro-F1**, subject to latency/memory constraints. For a real release, replace this random split with a locked creator/session-grouped test set.


In [ ]:
CANDIDATE_ENCODERS = [
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'intfloat/multilingual-e5-base',
    # Optional heavier candidate. Uncomment on a suitable GPU:
    # 'BAAI/bge-m3',
]

train_df, test_df = train_test_split(
    f, test_size=.30, random_state=SEED, stratify=f['intent']
)
le = LabelEncoder().fit(f['intent'])
y_train = le.transform(train_df.intent)
y_test = le.transform(test_df.intent)

results = []
models = {}
for model_name in CANDIDATE_ENCODERS:
    print('\nEncoding with', model_name)
    enc = SentenceTransformer(model_name)
    prefix = 'query: ' if 'e5' in model_name.lower() else ''
    X_train = enc.encode((prefix + train_df.text).tolist(), normalize_embeddings=True, show_progress_bar=True)
    X_test = enc.encode((prefix + test_df.text).tolist(), normalize_embeddings=True, show_progress_bar=True)
    clf = LogisticRegression(max_iter=4000, class_weight='balanced', C=4.0, random_state=SEED)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    macro = f1_score(y_test, pred, average='macro')
    results.append({'encoder': model_name, 'macro_f1': macro, 'dim': X_train.shape[1]})
    models[model_name] = (enc, clf, prefix)
    print(classification_report(y_test, pred, target_names=le.classes_, digits=3, zero_division=0))

bench = pd.DataFrame(results).sort_values('macro_f1', ascending=False)
display(bench)
BEST_ENCODER = bench.iloc[0].encoder
encoder, intent_clf, PREFIX = models[BEST_ENCODER]
print('Selected:', BEST_ENCODER)


## 4. Out-of-scope rejection

A premium assistant should be willing to say "I don't know the visitor's intent". The classifier therefore uses a probability threshold. Tune this threshold on a dedicated OOS validation set in production instead of forcing every message into a business intent.


In [ ]:
# Bootstrap OOS examples. Replace with a locked, human-labelled OOS validation file.
oos = [
    'what is the weather tomorrow', 'tell me a joke', 'who won the match',
    'quelle heure est-il', 'donne moi une recette', 'ما حالة الطقس اليوم',
    'من فاز بالمباراة', 'I am just saying hello'
]
X_oos = encoder.encode([(PREFIX + x) for x in oos], normalize_embeddings=True)
oos_max = intent_clf.predict_proba(X_oos).max(axis=1)
print(pd.DataFrame({'text': oos, 'max_probability': oos_max}))

# Conservative bootstrap threshold. In production choose it from validation curves.
OOS_THRESHOLD = max(0.55, float(np.quantile(oos_max, .90)))
print('Bootstrap OOS threshold:', round(OOS_THRESHOLD, 3))


## 5. Per-language evaluation

Macro-F1 can hide a weak language. We report each locale separately. Production release criteria should include a minimum per-language score and false-accept rate.


In [ ]:
X_test = encoder.encode([(PREFIX + x) for x in test_df.text], normalize_embeddings=True)
proba = intent_clf.predict_proba(X_test)
pred = proba.argmax(axis=1)
report_rows = []
for locale in sorted(test_df.locale.unique()):
    mask = test_df.locale.to_numpy() == locale
    report_rows.append({
        'locale': locale,
        'rows': int(mask.sum()),
        'macro_f1': f1_score(y_test[mask], pred[mask], average='macro')
    })
display(pd.DataFrame(report_rows))


## 6. Export deployable intent artifacts

The Python inference service in `services/ai/` loads these artifacts. The embedding model is referenced by model ID rather than copied into the ZIP.


In [ ]:
joblib.dump(intent_clf, ARTIFACTS / 'intent_classifier.joblib')
joblib.dump(le, ARTIFACTS / 'intent_labels.joblib')
meta = {
    'encoder': BEST_ENCODER,
    'prefix': PREFIX,
    'oos_threshold': OOS_THRESHOLD,
    'labels': le.classes_.tolist(),
    'bootstrap_rows': len(f),
    'warning': 'Bootstrap metrics are not production KPIs. Re-evaluate on a locked real-user test set.'
}
(ARTIFACTS / 'intent_meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
print(meta)


## 7. Semantic proof retrieval

Generation should be grounded in the creator's real work. Retrieve relevant projects/testimonials first, then give only those facts to the LLM. In production evaluate this on human relevance judgements using Recall@K and nDCG.


In [ ]:
def rank_proof(query, items, top_k=5):
    if not items: return []
    q = encoder.encode([PREFIX + query], normalize_embeddings=True)
    x = encoder.encode([(PREFIX + item) for item in items], normalize_embeddings=True)
    scores = cosine_similarity(q, x)[0]
    order = np.argsort(-scores)[:top_k]
    return [(items[i], float(scores[i])) for i in order]

example_items = [
    'Luxury residential interior redesign in Tunis',
    'B2B SaaS dashboard UX project',
    'Brand identity and packaging for a coffee shop',
]
rank_proof('I need a premium interior designer', example_items)


## 8. CTA/conversion ranker from real Faymoos events

This model is intentionally **not trained on fabricated conversions**. When `ai/data/faymoos_events.csv` exists, the cell trains a LightGBM rank/classification baseline. Use time-based or creator-grouped holdout in a real deployment. Optimize PR-AUC and calibration before online A/B uplift, not raw accuracy.


In [ ]:
from lightgbm import LGBMClassifier

EVENTS = DATA / 'faymoos_events.csv'
if EVENTS.exists():
    events = pd.read_csv(EVENTS)
    assert 'converted' in events.columns
    y = events.pop('converted').astype(int)
    X = pd.get_dummies(events, dummy_na=True)
    # Replace this split with time/creator-group holdout when production data is available.
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=SEED, stratify=y)
    ranker = LGBMClassifier(
        n_estimators=500, learning_rate=.035, num_leaves=31,
        subsample=.85, colsample_bytree=.85, random_state=SEED
    )
    ranker.fit(Xtr, ytr)
    p = ranker.predict_proba(Xte)[:,1]
    print('PR-AUC:', round(average_precision_score(yte, p), 4))
    print('ROC-AUC:', round(roc_auc_score(yte, p), 4))
    joblib.dump({'model': ranker, 'columns': X.columns.tolist()}, ARTIFACTS / 'cta_ranker.joblib')
else:
    print('No real event dataset found. This is deliberate: do not train a conversion model on invented outcomes.')


## 9. Grounded generation evaluation

Do not score a generative assistant with classifier "accuracy". Evaluate a fixed test set of creator contexts and requests with:

- **factual support**: every claimed client/result/credential exists in retrieved context,
- **hallucination rate**: unsupported factual claims / total factual claims,
- **human preference**: pairwise blinded rating for clarity, tone and usefulness,
- **task success**: whether the copy contains one unambiguous CTA when required,
- **business outcome**: online CTA/lead uplift after safety and quality gates pass.

The production Faymoos Copilot already receives creator portfolio/testimonial context and is instructed not to invent proof.


## 10. Smart Space vision evaluation

Object datasets such as ADE20K or SUN RGB-D can help later hotspot/object suggestions. They are not substitutes for panorama reconstruction data.

For the Smart Scan pipeline, benchmark:

- successful reconstruction rate,
- rescan rate,
- seam/artifact rate,
- geometric consistency on doors/walls/furniture,
- processing latency,
- user-rated fidelity,
- mobile-device compatibility.

Keep AI inpainting conservative. A system that "improves" a real room by inventing furniture is not accurate reconstruction.


## 11. Release gate

A new AI artifact is promoted only when it beats the current production artifact on the locked evaluation set **and** does not regress multilingual/OOS behaviour. Store model ID, dataset version, metrics and git commit with each artifact.

Recommended operational loop:

`collect consented feedback → redact/deduplicate → label → train → locked evaluation → shadow test → A/B test → promote/rollback`.
